In [16]:
!pip install google-cloud-bigquery-storage

In [17]:
from google.cloud import bigquery
from google.cloud import bigquery_storage  # ← should import without warning now

print("✓ BigQuery Storage module found!")

✓ BigQuery Storage module found!


In [18]:
"""
AgroSense Analytics — CSV to BigQuery ETL Pipeline
====================================================
Converted from MySQL → BigQuery (Free Tier Compatible)
 
Order of execution (respects FK constraints):
    1. dim_regions       — already in BigQuery (static seed)
    2. dim_farms         — from cleaned_smart_farming_df.csv
    3. dim_crops         — from cleaned_crop_rec_df.csv
    4. fact_yield        — from cleaned_smart_farming_df.csv
 
Setup:
    pip install google-cloud-bigquery pandas pyarrow db-dtypes numpy
 
Authentication (run once in terminal):
    gcloud auth application-default login
"""
 
# ── STEP 1: Imports ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
from google.cloud import bigquery
from datetime import datetime, timezone ,UTC
 
# ── STEP 2: Configuration ──────────────────────────────────────────────────────
PROJECT_ID = "agrosense-493415"   # ← Replace with your GCP project ID
DATASET_ID = "agrosense"
KEY_FILE     = "agrosense_key.json"       # path to your downloaded JSON key
YIELD_CSV  = "./cleaned_data/cleaned_smart_farming_df.csv"
REC_CSV    = "./cleaned_data/cleaned_crop_rec_df.csv"
 
# ── STEP 3: BigQuery Client ────────────────────────────────────────────────────
# No password needed — uses Google Cloud credentials automatically.
# If using a service account key file, uncomment the line below:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_FILE
 
client = bigquery.Client(project=PROJECT_ID)
 
def log(msg):
    print(f"  ✓ {msg}")
 
def section(title):
    print(f"\n{'='*55}\n  {title}\n{'='*55}")
 
# ── STEP 4: Helper — write DataFrame to BigQuery ───────────────────────────────
def bq_append(df: pd.DataFrame, table_name: str, chunk_size: int = 10_000):
    """
    Appends a DataFrame to a BigQuery table in chunks.
    Uses load_table_from_dataframe — does NOT consume query quota (free-tier safe).
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
 
    total_rows = len(df)
    loaded = 0
 
    for start in range(0, total_rows, chunk_size):
        chunk = df.iloc[start : start + chunk_size]
        job = client.load_table_from_dataframe(chunk, table_id, job_config=job_config)
        job.result()  # wait for upload to finish
        loaded += len(chunk)
        print(f"    Uploaded rows {start + 1}–{loaded} / {total_rows}")
 
    log(f"Loaded {total_rows} rows → {table_id}")
 
 
def bq_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return results as a DataFrame."""
    return client.query(sql).to_dataframe()
 
def utc_now_str():
    """
    Current UTC timestamp as string.
    Uses timezone-aware datetime — no deprecation warning.
    """
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
def get_next_id(table_name: str, id_column: str) -> int:
    """
    Gets the next available ID from a BigQuery table.
    Works like AUTO_INCREMENT in MySQL.
    Returns 1 if table is empty.
    """
    try:
        result = client.query(f"""
            SELECT COALESCE(MAX({id_column}), 0) AS max_id
            FROM `{PROJECT_ID}.{DATASET_ID}.{table_name}`
        """).to_dataframe()

        next_id = int(result["max_id"].iloc[0]) + 1
        print(f"  ✓ Next {id_column} for {table_name}: {next_id}")
        return next_id

    except Exception as e:
        print(f"  ⚠ Could not fetch max ID, starting from 1. Error: {e}")
        return 1
# ── STEP 5: Build region_id lookup map ────────────────────────────────────────
def build_region_map():
    section("Building region_id lookup map")
    region_df = bq_query(f"""
        SELECT region_id, region_name
        FROM `{PROJECT_ID}.{DATASET_ID}.dim_regions`
    """)
    region_map = dict(zip(region_df["region_name"], region_df["region_id"]))
    log(f"Region map: {region_map}")
    return region_map
 
 
# ── STEP 6: dim_farms ─────────────────────────────────────────────────────────
def load_dim_farms(df_yield, region_map):
    section("STEP 2 — Loading dim_farms")
 
    dim_farms = (
        df_yield[["farm_id", "region", "latitude", "longitude",
                  "irrigation_type", "fertilizer_type"]]
        .drop_duplicates("farm_id")
        .copy()
    )
 
    dim_farms["region_id"] = dim_farms["region"].map(region_map)
    dim_farms.drop(columns=["region"], inplace=True)
 
    dim_farms = dim_farms[[
        "farm_id", "region_id", "latitude", "longitude",
        "irrigation_type", "fertilizer_type"
    ]]
 
    # BigQuery type safety
    dim_farms["farm_id"]         = dim_farms["farm_id"].astype(str)
    dim_farms["region_id"]       = dim_farms["region_id"].astype("Int64")
    dim_farms["irrigation_type"] = dim_farms["irrigation_type"].astype(str)
    dim_farms["fertilizer_type"] = dim_farms["fertilizer_type"].astype(str)
 
    bq_append(dim_farms, "dim_farms")
    return dim_farms
 
 
# ── STEP 7: dim_crops ─────────────────────────────────────────────────────────
def load_dim_crops(df_rec, region_map):
    section("STEP 3 — Loading dim_crops")
 
    dim_crops = (
        df_rec
        .groupby(["label", "region"])
        .agg(
            N_avg              = ("N",           "mean"),
            N_min              = ("N",           "min"),
            N_max              = ("N",           "max"),
            P_avg              = ("P",           "mean"),
            P_min              = ("P",           "min"),
            P_max              = ("P",           "max"),
            K_avg              = ("K",           "mean"),
            K_min              = ("K",           "min"),
            K_max              = ("K",           "max"),
            ideal_temp_min     = ("temperature", "min"),
            ideal_temp_max     = ("temperature", "max"),
            ideal_humidity_min = ("humidity",    "min"),
            ideal_humidity_max = ("humidity",    "max"),
            ideal_ph_min       = ("soil_pH",     "min"),
            ideal_ph_max       = ("soil_pH",     "max"),
            ideal_rainfall_min = ("rainfall",    "min"),
            ideal_rainfall_max = ("rainfall",    "max"),
        )
        .round(2)
        .reset_index()
    )
    # ── ADD crop_id ──────────────────────────────────────────
    dim_crops.insert(0, "crop_id", range(1, len(dim_crops) + 1))
    # insert(0, ...) puts crop_id as the FIRST column
    dim_crops.rename(columns={"label": "crop_type"}, inplace=True)
    dim_crops["crop_type"] = dim_crops["crop_type"].str.title()
         
    dim_crops["crop_id"] = dim_crops["crop_id"]
    dim_crops["region_id"] = dim_crops["region"].map(region_map)
    dim_crops.drop(columns=["region"], inplace=True)
    dim_crops["region_id"] = dim_crops["region_id"].astype("Int64")
 
    bq_append(dim_crops, "dim_crops")
    log(f"Inserted {len(dim_crops)} rows into dim_crops (5 crops × 5 regions = 25)")
    return dim_crops

# ── STEP 9: fact_yield ────────────────────────────────────────────────────────
def load_fact_yield(df_yield, region_map):
    section("STEP 5 — Loading fact_yield")
 
    fact_yield = df_yield[[
        "farm_id", "crop_type", "region",
        "sowing_date", "harvest_date", "total_days",
        "yield_kg_per_hectare", "NDVI_index"
    ]].copy()
    # ── AUTO INCREMENT ──────────────────────────────────────
    next_id = get_next_id("fact_yield", "yield_id")
    fact_yield.insert(0, "yield_id", range(next_id, next_id + len(fact_yield)))
    fact_yield["yield_id"] = fact_yield["yield_id"].astype("Int64")
    # ────────────────────────────────────────────────────────
    # BigQuery DATE columns need Python date objects
    fact_yield["sowing_date"]  = pd.to_datetime(fact_yield["sowing_date"]).dt.date
    fact_yield["harvest_date"] = pd.to_datetime(fact_yield["harvest_date"]).dt.date
 
    fact_yield["region_id"] = fact_yield["region"].map(region_map)
    fact_yield.drop(columns=["region"], inplace=True)
    fact_yield.rename(columns={"yield_kg_per_hectare": "yield_kg_per_ha"}, inplace=True)
    fact_yield["region_id"] = fact_yield["region_id"].astype("Int64")
    fact_yield["created_at"] = pd.Timestamp.now(tz="UTC").to_pydatetime()
    fact_yield["created_at"] = pd.to_datetime(fact_yield["created_at"], utc=True)
    
    fact_yield = fact_yield[[
        "yield_id","farm_id", "crop_type", "region_id",
        "sowing_date", "harvest_date", "total_days",
        "yield_kg_per_ha", "NDVI_index","created_at"
    ]]
 
    bq_append(fact_yield, "fact_yield")
    return fact_yield
 
 
# ── STEP 10: Verification ─────────────────────────────────────────────────────
def verify_row_counts():
    section("Verification — Row Counts")
    tables = ["dim_regions", "dim_farms", "dim_crops","fact_yield"]
    for tbl in tables:
        result = bq_query(
            f"SELECT COUNT(*) AS cnt FROM `{PROJECT_ID}.{DATASET_ID}.{tbl}`"
        )
        count = result["cnt"].iloc[0]
        print(f"  {tbl:<30} {count:>8} rows")
 
 
# ── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n  AgroSense Analytics — BigQuery ETL Pipeline Starting...")
    print(f"  Project : {PROJECT_ID}   Dataset : {DATASET_ID}")
 
    # Test connection
    try:
        client.query("SELECT 1").result()
        log("BigQuery connection successful")
    except Exception as e:
        print(f"\n  ✗ Connection failed: {e}")
        print("  Check your PROJECT_ID and authentication.")
        exit(1)
 
    # Load CSVs
    section("Loading source CSV files")
    df_yield = pd.read_csv(YIELD_CSV)
    log(f"Yield CSV: {df_yield.shape[0]} rows × {df_yield.shape[1]} cols")
    df_rec = pd.read_csv(REC_CSV)
    log(f"Rec CSV  : {df_rec.shape[0]} rows × {df_rec.shape[1]} cols")
 
    # Build lookup
    region_map = build_region_map()
 
    # Load tables  (comment out any you don't want to run)
    #load_dim_farms(df_yield, region_map)
    #load_dim_crops(df_rec,   region_map)
    #load_fact_yield(df_yield, region_map)
 
    verify_row_counts()
    print("\n  ✓ ETL complete. All tables loaded successfully.\n")


  AgroSense Analytics — BigQuery ETL Pipeline Starting...
  Project : agrosense-493415   Dataset : agrosense
  ✓ BigQuery connection successful

  Loading source CSV files
  ✓ Yield CSV: 500 rows × 22 cols
  ✓ Rec CSV  : 2500 rows × 10 cols

  Building region_id lookup map
  ✓ Region map: {'North India': np.int64(1), 'South India': np.int64(2), 'Central USA': np.int64(3), 'South USA': np.int64(4), 'East Africa': np.int64(5)}

  STEP 2 — Loading dim_farms
    Uploaded rows 1–500 / 500
  ✓ Loaded 500 rows → agrosense-493415.agrosense.dim_farms

  STEP 3 — Loading dim_crops
    Uploaded rows 1–25 / 25
  ✓ Loaded 25 rows → agrosense-493415.agrosense.dim_crops
  ✓ Inserted 25 rows into dim_crops (5 crops × 5 regions = 25)

  STEP 5 — Loading fact_yield
  ✓ Next yield_id for fact_yield: 1
    Uploaded rows 1–500 / 500
  ✓ Loaded 500 rows → agrosense-493415.agrosense.fact_yield

  Verification — Row Counts
  dim_regions                           5 rows
  dim_farms                           5